# CNN14 hyperparameter search

Run the cells in order on a Google Colab GPU. This notebook compares fine-tuning settings on ESC-50 training folds 2?4 and validation fold 1. Test fold 5 is reserved and never scored here. Results are stored in Google Drive, so completed trials can be reused after a Colab disconnect.


In [ ]:
import importlib
import json
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive

project_dir = Path("/content/soundview")
panns_source_dir = Path("/content/audioset_tagging_cnn")
data_dir = Path("/content/datasets/ESC-50")

if project_dir.exists():
    subprocess.run(["git", "-C", str(project_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/AndreuLopezz/12607559_SoundView-environmental-sound-recognition.git",
            str(project_dir),
        ],
        check=True,
    )
if not panns_source_dir.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/qiuqiangkong/audioset_tagging_cnn.git",
            str(panns_source_dir),
        ],
        check=True,
    )
subprocess.run(
    [
        "git",
        "-C",
        str(panns_source_dir),
        "checkout",
        "d2f4b8c18eab44737fcc0de1248ae21eb43f6aa4",
    ],
    check=True,
)
if not data_dir.exists():
    data_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/karolpiczak/ESC-50.git",
            str(data_dir),
        ],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(project_dir / "requirements-panns.txt"),
    ],
    check=True,
)
sys.path.insert(0, str(project_dir))
drive.mount("/content/drive")
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before running the search")
device = torch.device("cuda")
print(f"GPU: {torch.cuda.get_device_name(device)}")

## Search setup

The completed float32 fold-5 run supplies the baseline validation score. Each new trial starts from the corresponding frozen fold-5 head and a fresh AudioSet CNN14. Only validation scores decide which settings continue to the next stage. The full 1,200 training clips are used because a small subset could rank learning rates incorrectly.


In [ ]:
from soundview import search_panns
from soundview.panns import load_panns_model

importlib.reload(search_panns)
from soundview.search_panns import (
    load_completed_validation_trial,
    run_panns_validation_trial,
)

DRIVE_DIR = Path("/content/drive/MyDrive/SoundView")
SEARCH_DIR = DRIVE_DIR / "panns_hyperparameters_v1"
PRETRAINED_PATH = DRIVE_DIR / "pretrained" / "Cnn14_mAP=0.431.pth"
FROZEN_HEAD_PATH = DRIVE_DIR / "panns_frozen_v1" / "fold5" / "best_head.pt"
BASELINE_PATH = DRIVE_DIR / "panns_full_fold5" / "fp32_v2" / "training_summary.json"

for required_path in [PRETRAINED_PATH, FROZEN_HEAD_PATH, BASELINE_PATH]:
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

baseline_summary = json.loads(BASELINE_PATH.read_text(encoding="utf-8"))
expected_baseline = {
    "epochs": 20,
    "batch_size": 2,
    "accumulation_steps": 4,
    "backbone_lr": 1e-6,
    "head_lr": 1e-5,
    "precision": "float32",
    "gradient_clip_norm": 1.0,
    "patience": 5,
    "seed": 42,
}
if (
    baseline_summary["test_fold_reserved"] != 5
    or baseline_summary["validation_fold"] != 1
):
    raise ValueError("The saved baseline belongs to a different ESC-50 split")
if baseline_summary["settings"] != expected_baseline:
    raise ValueError("The saved baseline used different fine-tuning settings")

BASE_CONFIG = {
    "backbone_lr": 1e-6,
    "head_lr": 1e-5,
    "weight_decay": 0.01,
    "head_dropout": 0.2,
}
BASELINE_F1 = baseline_summary["best_validation_macro_f1"]
print(f"Existing validation baseline: macro-F1 {BASELINE_F1:.4f}")


# Keep the training procedure fixed while the four searched settings change.
def trial_settings(config):
    return {
        "epochs": 20,
        "batch_size": 2,
        "accumulation_steps": 4,
        "backbone_lr": config["backbone_lr"],
        "head_lr": config["head_lr"],
        "weight_decay": config["weight_decay"],
        "head_dropout": config["head_dropout"],
        "patience": 5,
        "seed": 42,
        "precision": "float32",
        "gradient_clip_norm": 1.0,
    }


# A completed trial can be read from Drive; otherwise a fresh CNN14 is loaded.
def run_trial(name, config):
    trial_dir = SEARCH_DIR / name
    settings = trial_settings(config)
    previous = load_completed_validation_trial(trial_dir, 5, settings)
    if previous is not None:
        print(f"Reusing {name}: macro-F1 {previous['best_validation_macro_f1']:.4f}")
        return previous

    frozen_head = torch.load(FROZEN_HEAD_PATH, map_location="cpu", weights_only=True)
    backbone = load_panns_model(panns_source_dir, PRETRAINED_PATH, device)
    print(f"Starting {name}: {config}")
    try:
        return run_panns_validation_trial(
            backbone=backbone,
            head_checkpoint=frozen_head,
            data_dir=data_dir,
            output_dir=trial_dir,
            reserved_test_fold=5,
            backbone_lr=config["backbone_lr"],
            head_lr=config["head_lr"],
            weight_decay=config["weight_decay"],
            head_dropout=config["head_dropout"],
            device=device,
        )
    finally:
        del backbone
        torch.cuda.empty_cache()

## 1. Learning rates

Change one learning rate at a time while keeping the other settings at their baseline values. The baseline run is reused rather than trained again.


In [ ]:
best = {
    "name": "baseline",
    "config": BASE_CONFIG.copy(),
    "validation_macro_f1": BASELINE_F1,
}
trial_results = []
learning_rate_trials = [
    ("backbone_lr_3e-7", {"backbone_lr": 3e-7}),
    ("backbone_lr_3e-6", {"backbone_lr": 3e-6}),
    ("head_lr_3e-6", {"head_lr": 3e-6}),
    ("head_lr_3e-5", {"head_lr": 3e-5}),
]
for name, changes in learning_rate_trials:
    config = {**BASE_CONFIG, **changes}
    result = run_trial(name, config)
    score = result["best_validation_macro_f1"]
    trial_results.append(
        {
            "name": name,
            "config": config,
            "validation_macro_f1": score,
            "best_epoch": result["best_epoch"],
            "elapsed_seconds": result["elapsed_seconds"],
        }
    )
    if score > best["validation_macro_f1"]:
        best = {"name": name, "config": config, "validation_macro_f1": score}
print(
    f"Best learning rates: {best['name']}, validation macro-F1 {best['validation_macro_f1']:.4f}"
)

## 2. Weight decay

Keep the best learning rates and compare less or more regularization against AdamW's current value of 0.01.


In [ ]:
for name, weight_decay in [("weight_decay_0", 0.0), ("weight_decay_0_05", 0.05)]:
    config = {**best["config"], "weight_decay": weight_decay}
    result = run_trial(name, config)
    score = result["best_validation_macro_f1"]
    trial_results.append(
        {
            "name": name,
            "config": config,
            "validation_macro_f1": score,
            "best_epoch": result["best_epoch"],
            "elapsed_seconds": result["elapsed_seconds"],
        }
    )
    if score > best["validation_macro_f1"]:
        best = {"name": name, "config": config, "validation_macro_f1": score}
print(
    f"Best after weight decay: {best['name']}, validation macro-F1 {best['validation_macro_f1']:.4f}"
)

## 3. Classifier dropout

Change only the dropout applied before the ESC-50 classifier. The saved head weights are reused; dropout has no learned weights.


In [ ]:
for name, dropout in [("dropout_0", 0.0), ("dropout_0_4", 0.4)]:
    config = {**best["config"], "head_dropout": dropout}
    result = run_trial(name, config)
    score = result["best_validation_macro_f1"]
    trial_results.append(
        {
            "name": name,
            "config": config,
            "validation_macro_f1": score,
            "best_epoch": result["best_epoch"],
            "elapsed_seconds": result["elapsed_seconds"],
        }
    )
    if score > best["validation_macro_f1"]:
        best = {"name": name, "config": config, "validation_macro_f1": score}
print(f"Selected settings: {best['config']}")
print(f"Selected validation macro-F1: {best['validation_macro_f1']:.4f}")

## Search summary

The selected configuration is based only on validation macro-F1. This notebook does not score test fold 5 or create a five-fold test report. The selected settings can be evaluated later in a separate full experiment.


In [ ]:
summary = {
    "model": "panns_hyperparameter_search",
    "training_folds": [2, 3, 4],
    "validation_fold": 1,
    "test_fold_reserved": 5,
    "baseline_validation_macro_f1": BASELINE_F1,
    "baseline_config": BASE_CONFIG,
    "trials": trial_results,
    "selected": best,
}
SEARCH_DIR.mkdir(parents=True, exist_ok=True)
temporary_path = SEARCH_DIR / "search_summary.json.part"
temporary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
temporary_path.replace(SEARCH_DIR / "search_summary.json")
print(f"Compared {len(trial_results)} new trials; selected {best['name']}")
print(f"Summary saved to {SEARCH_DIR / 'search_summary.json'}")